### Client with SP permission, to add user to a group

- https://docs.databricks.com/aws/en/security/secrets/example-secret-workflow
- https://docs.databricks.com/aws/en/admin/users-groups/groups

In [0]:
from databricks.sdk import WorkspaceClient

# Retrieve SP credentials for API authentication
sp_client_id = dbutils.secrets.get(scope="mgmt-scope", key="sp-id")
sp_secret = dbutils.secrets.get(scope="mgmt-scope", key="sp-secret")

# Initialize client using the Service Principal
w = WorkspaceClient(
    host=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get(),
    client_id=sp_client_id,
    client_secret=sp_secret
)

# Granular Access: Add a user or another SP to a specific group
target_group_name = "data-analytics-team"
user_to_add = "analyst@example.com"

# Find group ID and add member
group = next(w.groups.list(filter=f"displayName eq '{target_group_name}'"))
w.groups.patch(
    id=group.id,
    operations=[{"op": "add", "path": "members", "value": [{"display": user_to_add}]}]
)


### Tables

In [0]:
tables = [
    "end_to_end_retail.db_landing.tbl_customer_autoloader",
    "end_to_end_retail.db_landing.tbl_customer_taxid",
    "end_to_end_retail.db_landing.tbl_sales",
    "end_to_end_retail.db_landing.tbl_sales_oders_ordered_product",
    "end_to_end_retail.db_landing.tbl_source_files_customers",
    "end_to_end_retail.db_landing.tbl_source_files_sales",
    "end_to_end_retail.db_landing.tbl_source_files_sales_orders",
    "end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched",
    "end_to_end_retail.db_bronze.tbb_customer_changes_daily",
    "end_to_end_retail.db_bronze.tbb_northstar_outfitters_orders"
]

In [0]:
%sql
-- Group 1 Creation and Membership
CREATE GROUP group_example;

- Creating just one user

In [0]:
from databricks.sdk import WorkspaceClient

# Authenticates using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

# Create a new user
new_user = w.users.create(
    display_name="user_one",
    user_name="user_one@example.com"
)

# print(f"Created user with ID: {new_user.id}")

- Creating multiple users

In [0]:
from databricks.sdk import WorkspaceClient

# Authenticates using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

# Create three new users
users_to_create = [
    {"display_name": "user_four", "user_name": "user_four@example.com"},
    {"display_name": "user_two", "user_name": "user_two@example.com"},
    {"display_name": "user_three", "user_name": "user_three@example.com"}
]

created_users = [w.users.create(**user) for user in users_to_create]

# print([user.id for user in created_users])

- Adding user to a group

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam

# Initializes using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

# 1. Provide the identifiers
group_name = "group_example"
user_email = "user_one@example.com"

# 2. Retrieve IDs for the group and user
# (In production, use w.groups.list(filter=...) or w.users.list(filter=...))
target_group = next(g for g in w.groups.list() if g.display_name == group_name)
target_user = next(u for u in w.users.list() if u.user_name == user_email)

# 3. Add the user to the group using a PATCH operation
w.groups.patch(
    id=target_group.id,
    operations=[
        iam.Patch(
            op=iam.PatchOp.ADD,
            value={
                "members": [
                    {
                        "value": target_user.id
                    }
                ]
            }
        )
    ],
    schemas=[iam.PatchSchema.URN_IETF_PARAMS_SCIM_API_MESSAGES_2_0_PATCH_OP]
)

print(f"Successfully added {user_email} to group {group_name}.")


- getting groups info

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get all groups and find the one at a specific index (e.g., index 0 for the 1st group)
all_groups = list(w.groups.list())
target_index = 0 

if len(all_groups) > target_index:
    group = all_groups[target_index]
    # print(f"Index {target_index} Group Name: {group.display_name}")
    # print(f"Group ID: {group.id}")

In [0]:
# Grant SELECT permission to group on a table
table_name = "end_to_end_retail.db_landing.tbl_customer_autoloader"
sql = f"GRANT SELECT ON TABLE {table_name} TO `group:{group_name}`"
spark.sql(sql)

### UC-Table-ACL

#### Granting permission to groups or users in tables with conditions

In [0]:
# Grant SELECT permission to a specific user on a table
table_name = "end_to_end_retail.db_landing.tbl_customer_autoloader"
user_email = "user_one@example.com"
sql = f"GRANT SELECT ON TABLE {table_name} TO `{user_email}`"
spark.sql(sql)

In [0]:
%sql
SHOW GRANTS ON TABLE end_to_end_retail.db_landing.tbl_customer_autoloader

In [0]:
%sql
-- Let's grant all users a SELECT
GRANT SELECT ON TABLE end_to_end_retail.db_landing.tbl_customer_autoloader TO `group_example`; -- skip it for the demo, 
GRANT SELECT, MODIFY ON TABLE customers TO `dataengineers`;

In [0]:
%sql
-- Access for Group 1 to Schema A
-- GRANT USE CATALOG ON CATALOG end_to_end_retail TO group_example;
-- GRANT USE SCHEMA ON SCHEMA end_to_end_retail.db_landing TO group_example;
-- GRANT SELECT ON SCHEMA end_to_end_retail.db_landing TO group_example;

#### Column Access Control Example

In [0]:
%sql
CREATE OR REPLACE FUNCTION end_to_end_retail.db_landing.pii_mask(column_value STRING)
RETURNS STRING
RETURN CASE
  WHEN current_user() = 'user_one@company.com' THEN column_value
  ELSE 'REDACTED'
END;


-- CREATE OR REPLACE FUNCTION main_catalog.default.pii_masking_policy(column_value STRING)
-- RETURN CASE
--   -- Check if the current user is the authorized PII viewer
--   WHEN is_account_group_member('pii_admins') OR current_user() = 'user3_authorized@company.com' 
--   THEN column_value
--   ELSE '***REDACTED***'
-- END;

In [0]:
%sql
-- Cleanup any row-filters or masks that may have been added in previous runs of the demo
-- ALTER TABLE customers DROP ROW FILTER;
-- ALTER TABLE customers ALTER COLUMN address DROP MASK;

In [0]:
%sql
SELECT 
  customer_id,
  end_to_end_retail.db_landing.pii_mask(email) AS masked_email
FROM end_to_end_retail.db_landing.tbl_customer_autoloader
limit 5

#### Row-level access control

In [0]:
%sql
-- get the current user (for informational purposes)
SELECT current_user(), is_account_group_member('account_example');

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 10

In [0]:
%sql
CREATE OR REPLACE FUNCTION region_filter(region_param STRING) 
RETURN 
  is_account_group_member('account_example') or  -- bu_admin can access all regions
  region_param like "NY%";                -- non bu_admin's can only access regions containing US

-- Grant access to all users to the function for the demo by making all account users owners.  Note: Don't do this in production!
-- GRANT ALL PRIVILEGES ON FUNCTION region_filter TO `account users`; 



In [0]:
%sql
-- Let's try our filter. As expected, we can access USA but not SPAIN.
SELECT region_filter('NY'), region_filter('GA')

In [0]:
%sql
ALTER TABLE end_to_end_retail.db_landing.tbl_customer_autoloader SET ROW FILTER region_filter ON (state);

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 15

In [0]:
%sql
SELECT DISTINCT(state) FROM end_to_end_retail.db_landing.tbl_customer_autoloader;

In [0]:
%sql
ALTER TABLE end_to_end_retail.db_landing.tbl_customer_autoloader DROP ROW FILTER;
-- Confirming that we can once again see all countries:
SELECT DISTINCT(state) FROM end_to_end_retail.db_landing.tbl_customer_autoloader;

In [0]:
# %sql
# CREATE OR REPLACE FUNCTION region_filter_dynamic(country_param STRING) 
# RETURN 
#   is_account_group_member('account_example') or                           -- bu_admin can access all regions
#   is_account_group_member(CONCAT('ANALYST_', country_param)); --regional admins can access only if the region (country column) matches the regional admin group suffix.
  
# -- apply the new access rule to the customers table:
# ALTER TABLE customers SET ROW FILTER region_filter_dynamic ON (country);

# SELECT region_filter_dynamic('USA'), region_filter_dynamic('SPAIN')

#### Column-level access control

#### Masking function

In [0]:
# Identify the Catalog and the Masking Function to use
catalog_name = "main_catalog"
mask_function = "main_catalog.default.pii_masking_policy"
target_tag = "PII"

# Query the Information Schema to find all columns with the specific tag
tagged_columns = spark.sql(f"""
    SELECT table_catalog, table_schema, table_name, column_name
    FROM {catalog_name}.information_schema.column_tags
    WHERE tag_name = '{target_tag}'
""")

for row in tagged_columns.collect():
    full_table_path = f"{row.table_catalog}.{row.table_schema}.{row.table_name}"
    column_name = row.column_name
    
    print(f"Applying mask to {full_table_path}.{column_name}")
    
    # Execute the SQL to set the mask
    spark.sql(f"""
        ALTER TABLE {full_table_path} 
        ALTER COLUMN {column_name} SET MASK {mask_function}
    """)
